<a href="https://colab.research.google.com/github/Ash100/DaS/blob/main/ESMFold_ProteinTTT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**ESMFold + ProteinTTT**

ProteinTTT: [Github](https://github.com/anton-bushuiev/ProteinTTT), [Hugging Face Space](https://huggingface.co/spaces/pimenova/ProteinTTT), [Paper](https://arxiv.org/abs/2411.02109) (ICLR 2026)


ProteinTTT enables customizing protein language models to one protein at a time for enhanced performance on challenging targets. This notebook provides a demo for protein structure prediction.

Insert your sequence and job name below under Configuration -> Protein, and then press Runtime -> Run all.

In [1]:
%%time
#@title ## Installation
#@markdown Install ESMFold, ProteinTTT and download weights (~3min)
version = "1"
model_name = "esmfold_v0.model" if version == "0" else "esmfold.model"
import os, time
if not os.path.isfile(model_name):
  # download esmfold params
  os.system("apt-get install aria2 -qq")
  os.system(f"aria2c -q -x 16 https://colabfold.steineggerlab.workers.dev/esm/{model_name} &")

  if not os.path.isfile("finished_install"):
    # install dependencies
    print("installing auxiliary dependencies...")
    os.system("pip install -q omegaconf pytorch_lightning biopython ml_collections einops py3Dmol modelcif")
    os.system("pip install -q git+https://github.com/NVIDIA/dllogger.git")

    print("installing openfold...")
    # install openfold
    os.system(f"pip install -q git+https://github.com/sokrypton/openfold.git")

    print("installing esmfold...")
    # install esmfold
    os.system(f"pip install -q git+https://github.com/sokrypton/esm.git")

    print("installing proteinttt...")
    # install ProteinTTT (biotite, tmtools are required dependencies not auto-installed)
    os.system("pip install -q biotite tmtools")
    os.system("pip install -q git+https://github.com/cloneofsimo/lora.git")
    os.system("pip install -q git+https://github.com/anton-bushuiev/ProteinTTT.git")

    os.system("touch finished_install")

  # wait for Params to finish downloading...
  while not os.path.isfile(model_name):
    time.sleep(5)
  if os.path.isfile(f"{model_name}.aria2"):
    print("downloading params...")
  while os.path.isfile(f"{model_name}.aria2"):
    time.sleep(5)

installing auxiliary dependencies...
installing openfold...
installing esmfold...
installing proteinttt...
CPU times: user 12.9 ms, sys: 3.21 ms, total: 16.1 ms
Wall time: 3min 9s


In [6]:
#@title ##Configuration { display-mode: "form" }

#@markdown ### Protein
sequence = "MQLQKYIDNFDIITIYPNIMENNTGYEQRKYDENDIKNIKKTLAKFNANEKHSTYTESFYRNLIVMNTDGNITRLKKNNKYDIINNHLLLLTKIEYLETEQIPSINPDFTRTITSISFSFNNINISIDKENNVYTIGISLLKGYSEEILNEIILLF" #@param {type:"string"}
jobname = "example" #@param {type:"string"}

#@markdown ---
#@markdown ### ESMFold settings
num_recycles = 12 #@param {type:"integer"}

#@markdown ---
#@markdown ### ProteinTTT hyperparameters
ttt_steps = 10 #@param {type:"integer"}
ttt_lr = 4e-4 #@param {type:"number"}
ttt_batch_size = 4 #@param ["1", "2", "4"] {type:"raw"}
ttt_lora_rank = 8 #@param {type:"integer"}
ttt_lora_alpha = 32.0 #@param {type:"number"}
ttt_seed = 0 #@param {type:"number"}

In [7]:
%%time
#@title ## Run **ESMFold** { display-mode: "form" }
from string import ascii_uppercase, ascii_lowercase
import hashlib, re, os
import numpy as np
import torch
from jax.tree_util import tree_map
import matplotlib.pyplot as plt
from scipy.special import softmax
import gc

def parse_output(output):
  pae = (output["aligned_confidence_probs"][0] * np.arange(64)).mean(-1) * 31
  plddt = output["plddt"][0,:,1]

  bins = np.append(0,np.linspace(2.3125,21.6875,63))
  sm_contacts = softmax(output["distogram_logits"],-1)[0]
  sm_contacts = sm_contacts[...,bins<8].sum(-1)
  xyz = output["positions"][-1,0,:,1]
  mask = output["atom37_atom_exists"][0,:,1] == 1
  o = {"pae":pae[mask,:][:,mask],
       "plddt":plddt[mask],
       "sm_contacts":sm_contacts[mask,:][:,mask],
       "xyz":xyz[mask]}
  return o

def get_hash(x): return hashlib.sha1(x.encode()).hexdigest()
alphabet_list = list(ascii_uppercase+ascii_lowercase)

def load_esmfold(seq_length):
  """Load a fresh ESMFold model from disk every time."""
  _g = globals()
  for _v in ("ttt_model", "model"):
    if _v in _g:
      del _g[_v]
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

  m = torch.load(model_name, weights_only=False)
  m.eval().cuda().requires_grad_(False)
  if seq_length > 700:
    m.set_chunk_size(64)
  else:
    m.set_chunk_size(128)
  return m

# Process sequence from Configuration cell
_jobname = re.sub(r'\W+', '', jobname)[:50]
_sequence = re.sub("[^A-Z]", "", sequence.upper())

ID = _jobname+"_"+get_hash(_sequence)[:5]
length = len(_sequence)
lengths = [length]

model = load_esmfold(length)

torch.cuda.empty_cache()
output = model.infer(_sequence,
                     num_recycles=num_recycles)

pdb_str = model.output_to_pdb(output)[0]
output = tree_map(lambda x: x.cpu().numpy(), output)
ptm = output["ptm"][0]
plddt = output["plddt"][0,...,1].mean()
O = parse_output(output)
print(f'\n--- ESMFold ---')
print(f'ptm: {ptm:.3f} plddt: {plddt:.3f}')
os.system(f"mkdir -p {ID}")
prefix = f"{ID}/ptm{ptm:.3f}_r{num_recycles}_default"
np.savetxt(f"{prefix}.pae.txt",O["pae"],"%.3f")
with open(f"{prefix}.pdb","w") as out:
  out.write(pdb_str)
with open("esmfold.pdb","w") as out:
  out.write(pdb_str)

baseline_pdb_str = pdb_str
baseline_ptm = ptm
baseline_plddt = plddt
baseline_O = O


--- ESMFold ---
ptm: 0.251 plddt: 37.725
CPU times: user 10 s, sys: 3.66 s, total: 13.7 s
Wall time: 13.7 s


In [8]:
%%time
#@title ##Run **ESMFold + ProteinTTT** { display-mode: "form" }

# Ensure lora_diffusion is available (needed for LoRA-based TTT)
try:
  from lora_diffusion.lora import inject_trainable_lora
except ImportError:
  import subprocess, importlib
  subprocess.check_call(["pip", "install", "-q", "git+https://github.com/cloneofsimo/lora.git"])
  # Reload proteinttt.base so it picks up the newly installed package
  import proteinttt.base as _pb
  importlib.reload(_pb)

from proteinttt.models.esmfold import ESMFoldTTT, DEFAULT_ESMFOLD_TTT_CFG
from proteinttt.base import TTTModule
import esm as _esm
import copy as _copy

model = load_esmfold(length)

ttt_cfg = _copy.deepcopy(DEFAULT_ESMFOLD_TTT_CFG)
ttt_cfg.steps = ttt_steps
ttt_cfg.lr = ttt_lr
ttt_cfg.batch_size = ttt_batch_size
ttt_cfg.lora_rank = ttt_lora_rank
ttt_cfg.lora_alpha = ttt_lora_alpha
ttt_cfg.seed = ttt_seed

ttt_cfg.initial_state_reset = False
# Disable full structure prediction at every TTT step (saves GPU memory).
ttt_cfg.eval_each_step = True
# Disable saving copies of the best state during training.
ttt_cfg.automatic_best_state_reset = False

print(f"Applying ProteinTTT with {ttt_steps} steps (in-place)...")
model.__class__ = ESMFoldTTT
TTTModule.__init__(model, ttt_cfg=ttt_cfg)
model.ttt_alphabet = _esm.Alphabet.from_architecture("ESM-1b")
model.ttt_batch_converter = model.ttt_alphabet.get_batch_converter()
model._ttt_temp_pdb_path = None

ttt_model = model
model_name_ = ""

print("Starting ProteinTTT...")
ttt_result = ttt_model.ttt(_sequence)

ttt_df = ttt_result["df"]
ttt_step_data = ttt_result["ttt_step_data"]
best_row = ttt_df.loc[ttt_df["plddt"].idxmax()]
best_step = int(best_row["step"])
best_plddt_eval = best_row["plddt"]

best_pdb_raw = ttt_step_data[best_step]["eval_step_preds"]["pdb"]
pdb_str = best_pdb_raw[0] if isinstance(best_pdb_raw, list) else best_pdb_raw

print(f'Best step: {best_step}  mean pLDDT (eval): {best_plddt_eval:.3f}')
print(f'\n--- Comparison ---')
print(f'ESMFold pLDDT: {baseline_plddt:.3f} -> ESMFold + ProteinTTT pLDDT: {best_plddt_eval:.3f} (delta: {best_plddt_eval - baseline_plddt:+.3f})')

torch.cuda.empty_cache()
ttt_model.eval()
if length > 700:
  ttt_model.set_chunk_size(64)
else:
  ttt_model.set_chunk_size(128)

with torch.no_grad():
  ttt_output = ttt_model.infer(_sequence,
                       num_recycles=num_recycles)

ttt_output = tree_map(lambda x: x.cpu().numpy(), ttt_output)
ptm = ttt_output["ptm"][0]
plddt = ttt_output["plddt"][0,...,1].mean()
O = parse_output(ttt_output)

# print(f'\npLDDT across TTT steps:')
# for _, r in ttt_df.iterrows():
#   tag = " ◀ best" if int(r["step"]) == best_step else ""
#   print(f'  step {int(r["step"]):3d}  pLDDT {r["plddt"]:.3f}{tag}')

with open("esmfold_proteinttt.pdb","w") as out:
  out.write(pdb_str)

Applying ProteinTTT with 10 steps (in-place)...
Starting ProteinTTT...
2026-03-06 15:19:28,817 | INFO | step: 0, accumulated_step: 0, loss: None, perplexity: None, ttt_step_time: 0.00000, score_seq_time: 0.00000, eval_step_time: 2.16947, plddt: 34.51561, tm_score: None, lddt: None


INFO:ttt_log:step: 0, accumulated_step: 0, loss: None, perplexity: None, ttt_step_time: 0.00000, score_seq_time: 0.00000, eval_step_time: 2.16947, plddt: 34.51561, tm_score: None, lddt: None


2026-03-06 15:19:31,933 | INFO | step: 1, accumulated_step: 4, loss: 2.58594, perplexity: None, ttt_step_time: 0.93339, score_seq_time: 0.00000, eval_step_time: 2.18061, plddt: 32.04778, tm_score: None, lddt: None


INFO:ttt_log:step: 1, accumulated_step: 4, loss: 2.58594, perplexity: None, ttt_step_time: 0.93339, score_seq_time: 0.00000, eval_step_time: 2.18061, plddt: 32.04778, tm_score: None, lddt: None


2026-03-06 15:19:35,014 | INFO | step: 2, accumulated_step: 8, loss: 2.55859, perplexity: None, ttt_step_time: 0.89263, score_seq_time: 0.00000, eval_step_time: 2.18735, plddt: 32.51414, tm_score: None, lddt: None


...

2026-03-06 15:19:56,805 | INFO | step: 9, accumulated_step: 36, loss: 1.68359, perplexity: None, ttt_step_time: 0.89962, score_seq_time: 0.00000, eval_step_time: 2.20124, plddt: 40.97022, tm_score: None, lddt: None


INFO:ttt_log:step: 9, accumulated_step: 36, loss: 1.68359, perplexity: None, ttt_step_time: 0.89962, score_seq_time: 0.00000, eval_step_time: 2.20124, plddt: 40.97022, tm_score: None, lddt: None


2026-03-06 15:19:59,906 | INFO | step: 10, accumulated_step: 40, loss: 1.36719, perplexity: None, ttt_step_time: 0.90895, score_seq_time: 0.00000, eval_step_time: 2.19037, plddt: 50.73154, tm_score: None, lddt: None


INFO:ttt_log:step: 10, accumulated_step: 40, loss: 1.36719, perplexity: None, ttt_step_time: 0.90895, score_seq_time: 0.00000, eval_step_time: 2.19037, plddt: 50.73154, tm_score: None, lddt: None


Best step: 10  mean pLDDT (eval): 50.732

--- Comparison ---
ESMFold pLDDT: 37.725 -> ESMFold + ProteinTTT pLDDT: 50.732 (delta: +13.007)
CPU times: user 48.6 s, sys: 3.3 s, total: 51.9 s
Wall time: 51.9 s


In [9]:
#@title ## Display structures (optional) { display-mode: "form" }
import py3Dmol
from IPython.display import display, HTML

# AlphaFold pLDDT color scheme
AF_COLORS = {95: '#0053D6', 80: '#65CBF3', 60: '#FFDB13', 25: '#FF7D45'}

def _bin_bfactors(pdb_str):
  """Bin B-factors into 4 pLDDT categories so py3Dmol can map them."""
  lines = []
  for line in pdb_str.splitlines():
    if line.startswith(('ATOM', 'HETATM')) and len(line) >= 66:
      try:
        b = float(line[60:66])
        if b > 90:   b_bin = 95.0
        elif b > 70: b_bin = 80.0
        elif b > 50: b_bin = 60.0
        else:        b_bin = 25.0
        line = line[:60] + f'{b_bin:6.2f}' + line[66:]
      except (ValueError, IndexError):
        pass
    lines.append(line)
  return '\n'.join(lines)

def show_pdb(pdb_str, show_sidechains=False, show_mainchains=False,
             size=(800,480), hbondCutoff=4.0):
  pdb_binned = _bin_bfactors(pdb_str)
  view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width=size[0], height=size[1])
  view.addModel(pdb_binned, 'pdb', {'hbondCutoff': hbondCutoff})
  view.setStyle({'cartoon': {'colorscheme': {'prop':'b','map': AF_COLORS}}})
  if show_sidechains:
    BB = ['C','O','N']
    view.addStyle({'and':[{'resn':["GLY","PRO"],'invert':True},{'atom':BB,'invert':True}]},
                  {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"GLY"},{'atom':'CA'}]},
                  {'sphere':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
    view.addStyle({'and':[{'resn':"PRO"},{'atom':['C','O'],'invert':True}]},
                  {'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  if show_mainchains:
    BB = ['C','O','N','CA']
    view.addStyle({'atom':BB},{'stick':{'colorscheme':f"WhiteCarbon",'radius':0.3}})
  view.zoomTo()
  return view

show_sidechain_atoms = False #@param {type:"boolean"}
show_backbone_atoms = False #@param {type:"boolean"}

# AlphaFold-style color legend
legend_html = """
<div style="display:flex; align-items:center; gap:18px; font-family:sans-serif; font-size:13px; margin:6px 0;">
  <span style="font-weight:bold;">pLDDT:</span>
  <span><span style="display:inline-block;width:14px;height:14px;background:#0053D6;border-radius:2px;vertical-align:middle;margin-right:4px;"></span>Very high (&gt;90)</span>
  <span><span style="display:inline-block;width:14px;height:14px;background:#65CBF3;border-radius:2px;vertical-align:middle;margin-right:4px;"></span>Confident (70–90)</span>
  <span><span style="display:inline-block;width:14px;height:14px;background:#FFDB13;border-radius:2px;vertical-align:middle;margin-right:4px;"></span>Low (50–70)</span>
  <span><span style="display:inline-block;width:14px;height:14px;background:#FF7D45;border-radius:2px;vertical-align:middle;margin-right:4px;"></span>Very low (&lt;50)</span>
</div>"""

display(HTML("<h3>ESMFold + ProteinTTT</h3>" + legend_html))
show_pdb(pdb_str,
         show_sidechains=show_sidechain_atoms,
         show_mainchains=show_backbone_atoms).show()

display(HTML("<h3>ESMFold</h3>" + legend_html))
show_pdb(baseline_pdb_str,
         show_sidechains=show_sidechain_atoms,
         show_mainchains=show_backbone_atoms).show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [ ]:
#@title ## Download predictions { display-mode: "form" }
import zipfile
zip_name = f"{ID}.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
  zf.write("esmfold.pdb")
  zf.write("esmfold_proteinttt.pdb")
print(f"Saved {zip_name}")
try:
  from google.colab import files
  files.download(zip_name)
except ImportError:
  pass

Saved example_A0A345MMR4_32a95.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>